# Wafer Fabrication Log

Registra un nuevo batch de wafers en el sistema de memorias.

**Pasos:**
1. Fotolitografía (spin coat, laser writing, developing)
2. Evaporación Cr/Au
3. Lift-off
4. Cutting en piezas

Este notebook crea una memoria de tipo `experiment` con `subtype=batch`.

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '../..')))
sys.path.insert(0, str(PROJECT_ROOT))

from src import db

db.init_db()
print('Database initialized at', db.DB_PATH)

## 1. Información del Wafer

Define los parámetros del wafer.

In [ ]:
# === MODIFICAR ESTOS VALORES ===
DATE = datetime.now().strftime('%Y-%m-%d')
WAFER_NUM = 1
SUBSTRATE = 'Kapton'  # Kapton, Si/SiO2, Glass
THICKNESS = '125um'  # 75um, 125um, 150um, 300nm SiO2
N_PIECES = 4
N_DEVICES_PER_PIECE = 3
NOTES = ''
# ==============================

wafer_name = f'{DATE}_W{WAFER_NUM}'
print(f'Wafer name: {wafer_name}')
print(f'Substrate: {SUBSTRATE} ({THICKNESS})')
print(f'Pieces: {N_PIECES} x {N_DEVICES_PER_PIECE} devices = {N_PIECES * N_DEVICES_PER_PIECE} total')

In [ ]:
# Crear memoria para este wafer batch
memory_id = db.create_memory(
    title=f'Wafer {wafer_name}',
    type='experiment',
    subtype='wafer',
    date=DATE,
    description=f'Fabricación de wafer con {N_PIECES} piezas, {N_DEVICES_PER_PIECE} dispositivos por pieza',
    substrate=SUBSTRATE,
    process='photolithography + evaporation + lift-off',
    custom_fields={
        'thickness': THICKNESS,
        'n_pieces': str(N_PIECES),
        'n_devices_per_piece': str(N_DEVICES_PER_PIECE),
        'total_devices': str(N_PIECES * N_DEVICES_PER_PIECE),
        'notes': NOTES,
    }
)

print(f'Created wafer memory with id={memory_id}')

## 2. Fotolitografía

Registra los parámetros de fotolitografía.

In [ ]:
# === MODIFICAR ESTOS VALORES ===
PHOTO_PARAMS = {
    'resist': 'S1813',
    'spin_rpm': 4000,
    'spin_time_s': 25,
    'anneal_temp_C': 75,
    'anneal_time_min': 1,
}
PHOTO_NOTES = 'Spin coat 4000rpm 25s, anneal 75C 1min'
# ==============================

# Agregar como campo personalizado
for key, value in PHOTO_PARAMS.items():
    db.add_field(memory_id, f'photo_{key}', str(value))

if PHOTO_NOTES:
    db.add_field(memory_id, 'photo_notes', PHOTO_NOTES)

print(f'Registered photolithography parameters for wafer {wafer_name}')

## 3. Evaporación

Registra evaporación Cr/Au (hecho por técnicos de cleanroom).

In [ ]:
# === MODIFICAR ESTOS VALORES ===
EVAP_PARAMS = {
    'cr_nm': 5,
    'au_nm': 40,
}
# ==============================

for key, value in EVAP_PARAMS.items():
    db.add_field(memory_id, f'evap_{key}', str(value))

print(f'Registered evaporation: {EVAP_PARAMS["cr_nm"]}nm Cr + {EVAP_PARAMS["au_nm"]}nm Au')

## 4. Lift-off

Registra parámetros de lift-off (3x Acetone + 3x IPA).

In [ ]:
# === MODIFICAR ESTOS VALORES ===
LIFTOFF_PARAMS = {
    'acetone_baths': 3,
    'acetone_time_min': 15,
    'ipa_baths': 3,
    'ipa_time_min': 15,
}
# ==============================

for key, value in LIFTOFF_PARAMS.items():
    db.add_field(memory_id, f'lift_off_{key}', str(value))

print(f'Registered lift-off parameters')

## 5. Fotos (opcional)

Si tomaste fotos del proceso, puedes referenciarlas.

In [ ]:
# Buscar fotos en data/photos/
photos_dir = PROJECT_ROOT / 'data' / 'photos'

if photos_dir.exists():
    # Buscar fotos recientes (últimos 30 días)
    from datetime import timedelta
    cutoff = datetime.now() - timedelta(days=30)
    
    photos = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        photos.extend(photos_dir.rglob(ext))
    
    # Filtrar fotos recientes
    recent_photos = [p for p in photos if p.stat().st_mtime > cutoff.timestamp()]
    recent_photos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    
    print(f'Found {len(recent_photos)} recent photos (last 30 days)')
    
    # Referenciar primeras 5 fotos
    for photo in recent_photos[:5]:
        rel_path = photo.relative_to(PROJECT_ROOT)
        db.add_reference(memory_id, 'photo', str(rel_path))
        print(f'  Added: {rel_path}')
else:
    print('No photos directory found')

## Resumen

Muestra la memoria creada con todos los campos y referencias.

In [ ]:
# Mostrar memoria completa
memory = db.get_memory(memory_id)

print(f'=== Wafer Memory {memory_id} ===')
print(f'Title: {memory["title"]}')
print(f'Type: {memory["type"]} ({memory["subtype"]})')
print(f'Date: {memory["date"]}')
print(f'Substrate: {memory["substrate"]}')
print(f'Process: {memory["process"]}')
print(f'Description: {memory["description"]}')

print(f'\nCustom fields ({len(memory["fields"])}):')
for f in memory['fields']:
    print(f'  - {f["field_name"]}: {f["field_value"]}')

print(f'\nReferences ({len(memory["references"])}):')
for r in memory['references']:
    print(f'  - {r["ref_type"]}: {r["ref_path"]}')

---
### Próximos pasos

- Continuar con `template_sam_bams.ipynb` para registrar SAM + BAMS
- Usar `template_ink_prep.ipynb` para preparar ink
- Usar `template_measurement.ipynb` para registrar mediciones